In [7]:
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
# import ipympl

In [8]:
# Define learn function
# deg is the highest degree of the function. Ex cubic function is 3
def learn(alpha, num_epoch, deg, x_train, y_train):
    # Check parameters
    if len(np.shape(y_train)) != 1:
        raise ValueError(f"Invalid shape for y_train. Should be a flat array but received shape {np.shape(y_train)}")
    if len(np.shape(x_train)) != 2:
        raise ValueError(f"Invalid shape for x_train. Should be a 2D array but received shape {np.shape(x_train)}")
    if type(alpha) != float:
        raise ValueError(f"Alpha should be a float but received a {type(alpha)}")
    if type(num_epoch) != int:
        raise ValueError(f"num_epoch should be an int but received a {type(num_epoch)}")
    if type(deg) != int:
        raise ValueError(f"deg should be an int but received a {type(deg)}")


    x_train = np.array(x_train)
    weight_history = []
    error_history = []

    # Each row of the weights corresponds to the degree (x^0, x^1, etc)
    # Each column correspoonds to an input variable column (first input column, second input column)
    weights_num_cols = np.shape(x_train)[1]
    weights_num_rows = deg + 1
    weights = np.zeros(shape=(weights_num_rows, weights_num_cols))
    w = 0
    while w < weights_num_cols and w < weights_num_rows:
        weights[w][w] = 1
        w = w + 1

    rand_adds = np.zeros_like(weights)
    rand_adds = np.random.random(np.shape(weights)) / 10 - 0.05
    weights = weights + rand_adds


    for j in tqdm(range(num_epoch)):
        # Calculate sum of loss
        loss_sum = 0
        for n in range(len(x_train)):
            exp_input_n = np.array([x_train[n] ** i for i in range(deg + 1)]) # The first row is for the bias value
            weighted_input_n = np.array(weights * exp_input_n)
            pred_val = np.sum(weighted_input_n)
            loss = (y_train[n] - pred_val) * exp_input_n # A vector of loss for each parameter (weight and bias)

            loss_sum = loss_sum + loss

        error_history.append(np.sum(loss_sum))
        # Update theta
        weights = weights + alpha * loss_sum
        weight_history.append(weights)


    return weights, weight_history, error_history, loss_sum


In [9]:
def printModelData(data_inputs, data_y, weights, title):
    plt.scatter(data_inputs, data_y)
    x_space = np.linspace(np.min(data_inputs), np.max(data_inputs))
    num_weights = weights.shape[1] #Or 1?
    print(f'Num weights is {num_weights}')
    sample = np.array([1, 3])
    print(f'Sample result is f([1, 3]) = {np.sum(weights * [[sample ** i] for i in range(num_weights)])}')
    print(f'Step 1: {[[sample ** i] for i in range(num_weights)]}')
    print(f'Step 2: {weights * [[sample ** i] for i in range(num_weights)]}')
    print(f'Sample result is f(1) = {np.sum(weights * [[1 ** i] for i in range(num_weights)])}')
    model_predictions_sing_var =  [np.sum(weights * [x ** i for i in range(num_weights)]) for x in x_space]
    model_predictions_all_var =  [np.sum(weights * [x ** i for i in range(num_weights)]) for x in data_inputs]



    deg = weights.size - 1 # Linear has degree of 1
    x_space = np.linspace([0,0], np.max[10, 10])
    exp_input_n = [[x ** i for i in range(deg + 1)] for x in x_space] # The first row is for the bias value
    exp_input_n = np.array(exp_input_n)
    print("\nExp input n", exp_input_n)
    weighted_input_n = np.array(weights * exp_input_n)
    print("Weighted Exp input n", weighted_input_n)

    # input_n = np.array([x_train[n] ** i for i in range(num_params)])
    pred_val = np.sum(weighted_input_n)

    # plt.plot(x_space, model_predictions)
    plt.title(title)
    plt.show()

In [10]:
def showHistory(history, title):
    history = np.array(history)
    plt.scatter(x=[i for i in range(0, len(history))], y=history)
    plt.title(title)
    plt.show()

In [11]:
def predict_with_weights(weights, x_value, deg):
    if len(np.shape(x_value)) != 1:
        raise ValueError("Invalid x values")
    if len(np.shape(weights)) != 2:
        raise ValueError("Invalid weights")
    if type(deg) != int:
        raise ValueError("Invalid deg")
    exp_input_n = np.array([x_value ** i for i in range(deg + 1)]) # The first row is for the bias value
    weighted_input_n = np.array(weights * exp_input_n)
    pred_val = np.sum(weighted_input_n)
    return pred_val

# Part 2

Gen AI: I used Chat GPT to understand what a basis function is when used in the context of Machine Learning.

In [16]:
#House ID,Bathrooms,Land Area,Living area,# Garages,# Rooms,# Bedrooms,Age of home,Price
data_formats_input = {'names': ('id', 'baths', 'land', 'living', 'garages', 'rooms','beds','age'), 'formats':('i2', 'f2', 'f4','f4', 'f4', 'i4', 'i4','i4')}
data_formats_output = {'names': ('id', 'baths', 'land', 'living', 'garages', 'rooms','beds','age', 'price'), 'formats':('i2', 'f2', 'f4','f4', 'f4', 'i4', 'i4','i4','f8')}
housing_train = np.loadtxt("train.csv", delimiter=',', skiprows=1, ndmin=2, dtype=float)
housing_test = np.loadtxt("test.csv", delimiter=',', skiprows=1, dtype=float, ndmin=2)

housing_train = np.array(housing_train)
housing_test = np.array(housing_test)
# Ensure that all inputs are in the range of 0 to 1
train_max = np.max(housing_train, axis=0)
train_min = np.abs(np.min(housing_train, axis=0))
housing_train = (housing_train + train_min) / train_max

housing_test = (housing_test + train_min) / train_max
housing_train_input = housing_train[:,0:8]
housing_train_output = housing_train[:,8]

In [13]:
# Ideal hyperparameters
weights, weight_history, error_history, final_loss = learn(0.001, 1000000, 1, housing_train_input, housing_train_output)

print("Weights")
print(weights)

n = 'land'

housing_train = np.array(housing_train)

print(f"Final loss: {final_loss}")
print("Error history")
print(error_history)

# Plot history of error
error_history = np.array(error_history)
x_space = np.array([i for i in range(0, len(error_history))])
print(np.shape(x_space))
print(np.shape(error_history))
plt.scatter(x_space, error_history)
plt.title(f"Error history")
plt.show()


# Plot history of error
error_history = np.array(error_history)
start_x = len(error_history) - len(error_history) // 10
x_space = np.array([i for i in range(start_x, len(error_history))])
print(np.shape(x_space))
print(np.shape(error_history))
plt.scatter(x_space, error_history[start_x:])
plt.title(f"Last 10% of Error History")
plt.show()

  1%|          | 10359/1000000 [00:04<06:35, 2504.19it/s]


KeyboardInterrupt: 

In [ ]:
data_formats = {'names': ('id', 'baths', 'land', 'living', 'garages', 'rooms','beds','age', 'price'), 'formats':('i2', 'f2', 'f4','f4', 'f4', 'i4', 'i4','i4','f8')}
housing_train = np.loadtxt("train.csv", delimiter=',', skiprows=1, dtype=data_formats)
housing_test = np.loadtxt("test.csv", delimiter=',', skiprows=1, dtype=data_formats)
#House ID,Bathrooms,Land Area,Living area,# Garages,# Rooms,# Bedrooms,Age of home,Price

housing_train_output = housing_train["price"]

# for n in data_formats["names"]:
#     if n == 'price':
#         continue
#     print("\n\n", n)
#     housing_train_input = housing_train[n]

#     weights, weight_history, error_history, final_loss = learn(0.00001, 200, 4, housing_train_input, housing_train_output)


#     plt.scatter(housing_test[n], housing_test["price"])
#     x_space = np.linspace(np.min(housing_test[n]), np.max(housing_test[n]))
#     num_weights = weights.size()
#     model_predictions =  [np.sum(weights * [x ** i for i in range(num_weights)]) for x in x_space]
#     plt.plot(x_space, model_predictions)
#     plt.title("Test data with Model")
#     plt.show()

#     # Plot history of error
#     error_history = np.array(error_history)
#     plt.scatter([i for i in range(0, len(error_history))], error_history)
#     plt.title(f"Error history")
#     plt.show()

